[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C60_Edge_Deployment_Consistency_Course/01_preprocess/01_preprocess_consistency.ipynb)

# 01 · 预处理一致性（resize 家族 / letterbox / BGR / 归一化 / 对拍工具）

目标：**不 import cv2、不 import PIL、不 import torch**，从零把它们的 resize 全部实现一遍，
然后量化「写着同一个词、做着不同的事」到底差多少、以及差到什么程度会掉 mAP。

本 notebook 你会亲手实现：
1. **三种坐标映射语义**（half_pixel / align_corners / asymmetric）并量化差异
2. **四种插值核**：nearest、bilinear（无抗锯齿）、INTER_AREA、PIL 式抗锯齿三角核
3. **证明「3 倍下采样时 bilinear 就是 nearest」**——逐位相等，不是近似
4. **混叠的正弦扫频实验**：周期 4px 的假信号如何以 100% 幅度穿过 bilinear
5. **OpenCV 式定点 bilinear**，区分「实现噪声」与「语义错误」
6. **letterbox 的六个自由度** + 正/逆变换的 round-trip 单测 + 四种经典错误写法
7. **一个四类合成 TSR 检测任务**：量化 resize 不一致 / BGR 弄反 / 归一化顺序错各掉多少 mAP
8. **预处理对拍工具**：逐阶段 diff + 二分定位 + 指纹分类 + 定位报告

> 心智模型：**resize 不是「缩放图片」，是「重采样一个信号」。**
> 采样定理不会因为你调的是 `cv2.resize` 就失效。

## 1 · 坐标映射：输出第 j 个像素对应输入的哪个坐标

三种语义，三种哲学：
- **half_pixel**：像素是**面积**，取覆盖区间的中心 —— `u = (j+0.5)·s − 0.5`
- **align_corners**：像素是**点**，首尾严格对齐 —— `u = j·(H_in−1)/(H_out−1)`
- **asymmetric**：只对齐左上角 —— `u = j·s`

In [ ]:
import numpy as np, math, time
rng = np.random.default_rng(60)
np.set_printoptions(precision=4, suppress=True, linewidth=140)

def coord_map(out_size, in_size, mode='half_pixel'):
    """返回长度 out_size 的数组：输出像素 j 对应的输入连续坐标 u(j)。"""
    j = np.arange(out_size, dtype=np.float64)
    s = in_size / out_size
    if mode == 'half_pixel':                       # OpenCV / PIL / torch(默认) / ONNX(默认)
        return (j + 0.5) * s - 0.5
    if mode == 'align_corners':                    # torch align_corners=True
        return np.zeros(1) if out_size == 1 else j * ((in_size - 1) / (out_size - 1))
    if mode == 'asymmetric':                       # **TensorRT IResizeLayer 的默认值**
        return j * s
    raise ValueError(mode)

print('4 → 8 上采样，三种语义给出的采样坐标：')
for m in ['half_pixel', 'align_corners', 'asymmetric']:
    print(f'  {m:<16s}', coord_map(8, 4, m))

assert np.allclose(coord_map(8, 4, 'half_pixel'),
                   [-0.25, 0.25, 0.75, 1.25, 1.75, 2.25, 2.75, 3.25])
assert np.allclose(coord_map(8, 4, 'align_corners'), np.arange(8) * 3 / 7)
assert np.allclose(coord_map(8, 4, 'asymmetric'), np.arange(8) * 0.5)
# half_pixel 会越界（-0.25），所以实现里必须 clamp —— 这就是「diff 集中在边缘带」的来源
assert coord_map(8, 4, 'half_pixel')[0] < 0
# half_pixel 与 asymmetric 只差一个常数平移 (s-1)/2
s = 4 / 8
assert np.allclose(coord_map(8, 4, 'half_pixel') - coord_map(8, 4, 'asymmetric'), (s - 1) / 2)
print(f'\n✅ half_pixel − asymmetric ≡ (s−1)/2 = {(s-1)/2:.3f}（常数平移）')
print('   而 align_corners 的**斜率**不同 → 差异随位置增长 → diff 集中在边缘带。')

## 2 · 从零实现 nearest / bilinear

`resize_bilinear` 是本 notebook 的主力函数。注意 clamp 的位置——
越界坐标要 clamp 到 `[0, in−1]`，但**小数部分 frac 要在 clamp 之前算**
（这正是各家实现最容易分歧的一行）。

In [ ]:
def _bilinear_idx(out_size, in_size, mode):
    src = coord_map(out_size, in_size, mode)
    x0 = np.floor(src).astype(np.int64)
    frac = src - x0                                 # **frac 在 clamp 之前算**
    return np.clip(x0, 0, in_size-1), np.clip(x0+1, 0, in_size-1), frac

def resize_bilinear(img, oh, ow, mode='half_pixel'):
    img = np.asarray(img, dtype=np.float64)
    H, W = img.shape[:2]
    y0, y1, fy = _bilinear_idx(oh, H, mode)
    x0, x1, fx = _bilinear_idx(ow, W, mode)
    a, b = img[y0][:, x0], img[y0][:, x1]
    c, d = img[y1][:, x0], img[y1][:, x1]
    fx_, fy_ = fx[None, :], fy[:, None]
    if img.ndim == 3:
        fx_, fy_ = fx_[..., None], fy_[..., None]
    return (a*(1-fx_) + b*fx_)*(1-fy_) + (c*(1-fx_) + d*fx_)*fy_

def resize_nearest(img, oh, ow, mode='asymmetric'):
    img = np.asarray(img, dtype=np.float64)
    H, W = img.shape[:2]
    sy = np.clip(np.floor(coord_map(oh, H, mode)).astype(int), 0, H-1)
    sx = np.clip(np.floor(coord_map(ow, W, mode)).astype(int), 0, W-1)
    return img[sy][:, sx]

# —— 手算对拍：2x2 → 1x4（只放大宽度），答案可以口算 ——
tiny = np.array([[0., 10.]])
assert np.allclose(resize_bilinear(tiny, 1, 4, 'align_corners'), [[0, 10/3, 20/3, 10]])
assert np.allclose(resize_bilinear(tiny, 1, 4, 'half_pixel'),   [[0, 2.5, 7.5, 10]])
assert np.allclose(resize_bilinear(tiny, 1, 4, 'asymmetric'),   [[0, 5.0, 10, 10]])
assert np.allclose(resize_nearest(tiny, 1, 4), [[0, 0, 10, 10]])
# 常数图在任何模式下都必须保持常数（这是最基本的健全性检查）
const = np.full((7, 11), 137.0)
for m in ['half_pixel', 'align_corners', 'asymmetric']:
    assert np.allclose(resize_bilinear(const, 3, 5, m), 137.0), m
print('half_pixel   2→4 :', resize_bilinear(tiny, 1, 4, 'half_pixel')[0])
print('align_corners2→4 :', resize_bilinear(tiny, 1, 4, 'align_corners')[0])
print('asymmetric   2→4 :', resize_bilinear(tiny, 1, 4, 'asymmetric')[0])
print('nearest      2→4 :', resize_nearest(tiny, 1, 4)[0])
print('\n✅ 三种语义在同一段数据上给出三组不同的值 —— 差异是**语义**的，不是精度的。')

In [ ]:
# —— align_corners 的差异在一张自然图上有多大 ——
MEAN = np.array([123.675, 116.28, 103.53]); STD = np.array([58.395, 57.12, 57.375])

yy, xx = np.mgrid[0:64, 0:64]
nat = 128 + 100*np.sin(xx/9.0)*np.cos(yy/7.0)          # 平滑自然图（无细纹理）

hp = resize_bilinear(nat, 32, 32, 'half_pixel')
ac = resize_bilinear(nat, 32, 32, 'align_corners')
asym = resize_bilinear(nat, 32, 32, 'asymmetric')

d_ac = np.abs(hp - ac); d_as = np.abs(hp - asym)
print(f"{'比较':<28s}{'max(灰阶)':>12s}{'mean(灰阶)':>12s}{'mean(tensor)':>14s}")
for nm, d in [('half_pixel vs align_corners', d_ac), ('half_pixel vs asymmetric', d_as)]:
    print(f'{nm:<28s}{d.max():>12.3f}{d.mean():>12.3f}{d.mean()/STD.mean():>14.4f}')

# fp16 的表示误差作为对照
t = (nat - MEAN.mean())/STD.mean()
fp16_err = np.abs(t.astype(np.float32) - t.astype(np.float16).astype(np.float32)).max()
ratio = (d_ac.mean()/STD.mean()) / fp16_err
print(f'\nfp16 表示误差(max) = {fp16_err:.6f}')
print(f'align_corners 差异是它的 {ratio:.0f} 倍 —— 任何合理的容差都会判 FAIL')

assert 6.0 < d_ac.max() < 6.5 and 2.0 < d_ac.mean() < 2.4, (d_ac.max(), d_ac.mean())
assert 0.035 < d_ac.mean()/STD.mean() < 0.041
assert ratio > 30
# 边缘带 vs 内部：align_corners 的差异集中在边缘（斜率不同 → 位置相关）
edge = np.zeros((32, 32), bool); edge[:3] = edge[-3:] = True; edge[:, :3] = edge[:, -3:] = True
print(f'边缘带平均差 {d_ac[edge].mean():.3f}  vs  内部平均差 {d_ac[~edge].mean():.3f}')
assert d_ac[edge].mean() > d_ac[~edge].mean() * 1.5
print('✅ 边缘带的差异明显更大 —— 这就是「diff 集中在边缘 → 查 align_corners」这条指纹。')

## 3 · 整数倍缩放的陷阱：half_pixel 下 3× bilinear **就是** nearest

把 half_pixel 代入整数缩放比 s：

    u(j) = (j+0.5)·s − 0.5 = s·j + (s−1)/2

**s 为奇数时 (s−1)/2 是整数 ⇒ u(j) 恒为整数 ⇒ frac ≡ 0 ⇒ 权重 (1,0) ⇒ 退化成最近邻。**

而 1920→640 与 1080→360 都是 s=3。这是车端最常见的一条预处理。

In [ ]:
for s in [2, 3, 4, 5, 6]:
    H = 60*s
    frac = coord_map(60, H, 'half_pixel') - np.floor(coord_map(60, H, 'half_pixel'))
    used = 1.0/(s*s) if s % 2 == 1 else 2.0*2.0/(s*s)      # 参与计算的源像素比例（2D）
    tag = '**退化为 nearest**' if np.allclose(frac, 0) else f'frac ≡ {frac[0]:.2f}'
    print(f's={s}:  (s-1)/2 = {(s-1)/2:>4.1f}   {tag:<22s}  参与计算的源像素 ≈ {used:>6.1%}')
    if s % 2 == 1:
        assert np.allclose(frac, 0), s
    else:
        assert np.allclose(frac, 0.5), s

print('\n—— 在真实分辨率上验证：1920×1080 → 640×360 (s=3) ——')
big = np.clip(128 + 60*np.sin(np.mgrid[0:360, 0:480][1]/7.0)
              + 40*np.sin(np.mgrid[0:360, 0:480][0]/3.0), 0, 255)   # 用 480x360 代表原图
out_bl = resize_bilinear(big, 120, 160, 'half_pixel')               # 3 倍下采样
out_nn = big[1::3, 1::3]                                            # 直接抽样 u=3j+1
print('bilinear(3×) 与「直接抽 u=3j+1」的最大差 =', np.abs(out_bl - out_nn).max())
assert np.abs(out_bl - out_nn).max() == 0.0, '必须**逐位相等**，不是近似'
print('✅ 逐位相等 —— 3 倍下采样时，双线性没有做任何平均，它只是在抽样。')
print(f'   9 个源像素里只有 1 个被用到，{1-1/9:.1%} 的信息被直接丢弃。')

## 4 · 抗锯齿：INTER_AREA 与 PIL 式三角核

正确的下采样要求滤波器的支撑集在**输入坐标系**下是 ±s（而不是固定的 ±1）：
- **INTER_AREA**：盒滤波，输出像素 j 覆盖源区间 `[j·s, (j+1)·s)`，权重 = 重叠长度
- **PIL / antialias=True**：三角核，`filterscale = max(1, s)`，支撑集 ±filterscale

In [ ]:
def _area_w(out_size, in_size):
    """OpenCV INTER_AREA 的权重矩阵 (out_size, in_size)，每行和为 1。"""
    s = in_size / out_size
    W = np.zeros((out_size, in_size))
    for j in range(out_size):
        lo, hi = j*s, (j+1)*s
        for i in range(int(np.floor(lo)), min(int(np.ceil(hi)), in_size)):
            W[j, i] = max(0.0, min(hi, i+1) - max(lo, i))
    return W / W.sum(1, keepdims=True)

def _tri_w(out_size, in_size):
    """PIL / antialias=True 的三角(bilinear)核：支撑集随缩放比拉伸。"""
    s = in_size / out_size
    fscale = max(1.0, s); support = 1.0 * fscale
    W = np.zeros((out_size, in_size))
    for j in range(out_size):
        c = (j + 0.5) * s
        for i in range(max(0, int(np.floor(c-support+0.5))),
                       min(in_size, int(np.ceil(c+support+0.5)))):
            W[j, i] = max(0.0, 1.0 - abs((i + 0.5 - c) / fscale))
    return W / W.sum(1, keepdims=True)

def _sep(img, Wy, Wx):
    tmp = np.tensordot(Wy, img, axes=([1], [0]))        # (oh, W, ...)
    return np.swapaxes(np.tensordot(Wx, tmp, axes=([1], [1])), 0, 1)

def resize_area(img, oh, ow):
    img = np.asarray(img, float); H, W = img.shape[:2]
    return _sep(img, _area_w(oh, H), _area_w(ow, W))

def resize_antialias(img, oh, ow):
    img = np.asarray(img, float); H, W = img.shape[:2]
    return _sep(img, _tri_w(oh, H), _tri_w(ow, W))

# 健全性：权重每行和为 1；常数图保持常数；整数倍时 area = 块平均
for f in (_area_w, _tri_w):
    assert np.allclose(f(7, 21).sum(1), 1.0)
assert np.allclose(resize_area(np.full((9, 12), 5.0), 3, 4), 5.0)
assert np.allclose(resize_antialias(np.full((9, 12), 5.0), 3, 4), 5.0)
blk = np.arange(36.).reshape(6, 6)
assert np.allclose(resize_area(blk, 3, 3), blk.reshape(3, 2, 3, 2).mean((1, 3)))
print('AREA 在 2× 下采样时严格等于 2×2 块平均：')
print(resize_area(blk, 3, 3))
print('\n✅ 三个 resize 实现就位：resize_bilinear / resize_area / resize_antialias')

In [ ]:
# —— 混叠：正弦扫频（3 倍下采样，源幅度 100）——
x = np.arange(96)
print('输出端可表示的最细周期 = 2·s = 6 px（原图坐标）。比它细的一切都是假的。\n')
print(f"{'原图周期':>10s}{'可表示?':>9s}{'bilinear':>11s}{'AREA':>9s}{'抗锯齿':>9s}")
res = {}
for period in [3, 4, 5, 6, 8, 12, 24]:
    sig = np.tile(128 + 100*np.sin(2*np.pi*x/period), (6, 1))
    amp = lambda a: (a.max() - a.min())/2
    b = amp(resize_bilinear(sig, 2, 32, 'half_pixel')[0])
    a_ = amp(resize_area(sig, 2, 32)[0])
    t_ = amp(resize_antialias(sig, 2, 32)[0])
    res[period] = (b, a_, t_)
    ok = '✅' if period >= 6 else '❌'
    print(f'{period:>10d}{ok:>8s}{b:>11.1f}{a_:>9.1f}{t_:>9.1f}')

b4, a4, t4 = res[4]
assert b4 > 99, '周期 4px 在输出端根本不可表示，bilinear 却原样通过 -> 100% 假信号'
assert a4 < 40 and t4 < 25, (a4, t4)
b5, a5, t5 = res[5]
assert b5 > 90 and a5 < 60
assert res[3][0] < 1e-6, '周期 3px 恰好等于采样步长 -> 被抽成常数'
print('\n⚠️  周期 4px：输出端**不可能存在**这个结构，bilinear 却让它以 100% 幅度通过 ——')
print('   它在输出图上变成了一条凭空捏造的粗条纹。AREA 压到 33，抗锯齿三角核压到 18。')
print('✅ TSR 含义：1920→640 后，原图上笔画宽度 < 3px 的一切（限速数字的笔画！）都是假的。')

In [ ]:
# —— 在一张带细纹理的自然图上：480 → 160（3 倍）——
YY, XX = np.mgrid[0:480, 0:480]
im = np.clip(120 + 60*np.sin(XX/40.)*np.cos(YY/33.)
             + 45*np.sin(2*np.pi*XX/5.) + 45*np.sin(2*np.pi*YY/4.), 0, 255)
B = resize_bilinear(im, 160, 160, 'half_pixel')
A = resize_area(im, 160, 160)
T = resize_antialias(im, 160, 160)

print(f"{'':<22s}{'输出 std':>10s}{'与 AREA 的 mean|Δ|':>20s}{'max|Δ|':>10s}")
print(f"{'原图':<22s}{im.std():>10.2f}{'—':>20s}{'—':>10s}")
for nm, o in [('无抗锯齿 bilinear', B), ('INTER_AREA', A), ('抗锯齿 bilinear', T)]:
    print(f'{nm:<22s}{o.std():>10.2f}{np.abs(o-A).mean():>20.2f}{np.abs(o-A).max():>10.2f}')

# bilinear 在 3× 下就是抽样 —— 逐位相等
assert np.abs(B - im[1::3, 1::3]).max() == 0.0
assert abs(B.std() - im.std()) < 0.5, '抽样不改变高频能量 -> std 几乎不变'
mean_d, max_d = np.abs(B - A).mean(), np.abs(B - A).max()
print(f'\n无抗锯齿 vs AREA: mean {mean_d:.2f} 灰阶, max {max_d:.2f} 灰阶'
      f'  → {mean_d/STD.mean():.3f} tensor 单位')
assert 20 < mean_d < 23 and 45 < max_d < 55, (mean_d, max_d)
assert np.abs(T - A).mean() < 10, '两个都抗锯齿的实现之间差异小得多'
print('✅ 分水岭是「有没有抗锯齿」，不是「用了哪家库」：')
print(f'   无抗锯齿 vs AREA = {mean_d:.1f} 灰阶；两种抗锯齿实现之间只差 {np.abs(T-A).mean():.1f} 灰阶。')
print(f'⚠️  {mean_d/STD.mean():.2f} tensor 单位 = 模块 00 容差表里「resize 语义错误」那一档。')

## 5 · 定点实现噪声 vs 语义错误：容差放在哪

OpenCV 对 8 位图走**定点**路径：插值权重量化到 1/2048（`INTER_RESIZE_COEF_BITS=11`），
结果四舍五入回 uint8。**即使语义完全一致，它和一个 float64 教科书实现之间也有差。**
知道这个差有多大，你才敢设容差。

In [ ]:
def resize_bilinear_fixed(img, oh, ow, bits=11):
    """OpenCV 式：权重量化到 1/2^bits，输出四舍五入回整数。"""
    img = np.asarray(img, float); H, W = img.shape[:2]
    y0, y1, fy = _bilinear_idx(oh, H, 'half_pixel')
    x0, x1, fx = _bilinear_idx(ow, W, 'half_pixel')
    S = 1 << bits
    fxq = np.round(np.clip(fx, 0, 1)*S)/S
    fyq = np.round(np.clip(fy, 0, 1)*S)/S
    a, b = img[y0][:, x0], img[y0][:, x1]
    c, d = img[y1][:, x0], img[y1][:, x1]
    top = a*(1-fxq[None, :]) + b*fxq[None, :]
    bot = c*(1-fxq[None, :]) + d*fxq[None, :]
    return np.round(top*(1-fyq[:, None]) + bot*fyq[:, None])

Bf = resize_bilinear_fixed(im, 200, 200)          # 用非整数倍缩放，让 frac 真的起作用
Bx = resize_bilinear(im, 200, 200, 'half_pixel')
dfix = np.abs(Bf - Bx)
print(f'定点 vs float64（同一语义）: max {dfix.max():.4f}  mean {dfix.mean():.4f} 灰阶')
print(f'超过 1 灰阶的像素比例: {(dfix > 1).mean():.6%}')
assert dfix.max() < 0.6 and dfix.mean() < 0.3   # 0.5 来自取整，多出的一点来自权重量化
assert (dfix > 1).mean() == 0.0

print(f"\n{'差异来源':<26s}{'tensor 单位':>14s}{'该不该报警':>12s}")
LEDGER = [('fp16 表示误差',          fp16_err,                    '❌ 不报'),
          ('定点舍入(max≈0.52 灰阶)', dfix.max()/STD.mean(),       '❌ 不报'),
          ('align_corners 语义不同', d_ac.mean()/STD.mean(),      '✅ 报'),
          ('抗锯齿缺失(AREA↔LINEAR)', mean_d/STD.mean(),          '✅ 报')]
for nm, v, w in LEDGER:
    print(f'{nm:<26s}{v:>14.5f}{w:>12s}')
TOL_MAX, TOL_MEAN = 0.02, 0.005
gap_lo, gap_hi = LEDGER[1][1], LEDGER[2][1]
print(f'\n最窄的一道缝：{gap_lo:.4f} ~ {gap_hi:.4f}（{gap_hi/gap_lo:.1f} 倍），'
      f'阈值 max<={TOL_MAX} 正好卡在中间。')
assert gap_lo < TOL_MAX < gap_hi, (gap_lo, gap_hi)
assert LEDGER[0][1] < TOL_MAX and LEDGER[3][1] > TOL_MAX
print('✅ 同一组阈值：放过 fp16 与定点舍入，拦住 align_corners 与抗锯齿缺失。')

## 6 · letterbox 的六个自由度

1) pad 值 2) pad 位置（居中/左上） 3) pad 到哪（方形 / stride 倍数）
4) 缩放比取整 5) 是否允许放大 6) pad 的奇偶分配

先算一笔账：车载相机是 16:9 的，`1920×1080 → 640` 时两种方案的算力差异。

In [ ]:
def letterbox(h0, w0, target=640, pad_value=114, center=True, stride=None, scaleup=True):
    """只算几何参数，不真的填像素 —— 一致性问题全在这些参数里。"""
    r = min(target/h0, target/w0)
    if not scaleup:
        r = min(r, 1.0)
    nw, nh = int(round(w0*r)), int(round(h0*r))
    dw, dh = target-nw, target-nh
    if stride:                                   # YOLOv5 的 auto=True：只 pad 到 stride 倍数
        dw, dh = dw % stride, dh % stride
    left, top = (dw//2, dh//2) if center else (0, 0)
    oh, ow = nh+dh, nw+dw
    return dict(r=r, new=(nh, nw), top=top, left=left, out=(oh, ow),
                pad_value=pad_value, pad_frac=1 - (nh*nw)/(oh*ow))

H0, W0 = 1080, 1920
variants = [
    ('A 方形 pad, 居中, 114',   dict(stride=None, center=True,  pad_value=114)),
    ('B 方形 pad, 左上, 114',   dict(stride=None, center=False, pad_value=114)),
    ('C 方形 pad, 居中, 0',     dict(stride=None, center=True,  pad_value=0)),
    ('D stride32, 居中, 114',   dict(stride=32,   center=True,  pad_value=114)),
    ('E stride64, 居中, 114',   dict(stride=64,   center=True,  pad_value=114)),
]
print(f"{'变体':<24s}{'r':>8s}{'输出形状':>14s}{'上 pad':>8s}{'灰边占比':>10s}")
LB = {}
for nm, kw in variants:
    lb = letterbox(H0, W0, 640, **kw); LB[nm[0]] = lb
    print(f"{nm:<24s}{lb['r']:>8.4f}{str(lb['out']):>14s}{lb['top']:>8d}{lb['pad_frac']:>10.2%}")

assert LB['A']['out'] == (640, 640) and LB['A']['top'] == 140
assert abs(LB['A']['pad_frac'] - 0.4375) < 1e-9, LB['A']['pad_frac']
assert LB['D']['out'] == (384, 640) and abs(LB['D']['pad_frac'] - 0.0625) < 1e-9
assert LB['E']['out'] == (384, 640)          # 280 % 64 == 24 == 280 % 32，两者巧合相同
assert LB['B']['top'] == 0
saved = 1 - (384*640)/(640*640)
print(f'\n✅ 方形 pad 有 {LB["A"]["pad_frac"]:.2%} 的张量是灰边 —— {LB["A"]["pad_frac"]:.2%} 的卷积在算灰色。')
print(f'   换成 stride 倍数 pad：640×384，省下 {saved:.1%} 的推理算力，几乎白送。')
print('⚠️  代价是输入形状不再固定 → TensorRT 需要动态 shape 的 optimization profile（模块 02）。')
print('   很多量产系统仍用方形 pad，不是没算过账，是不想引入动态 shape 的复杂度。')

In [ ]:
# —— 正/逆变换：round-trip 单测 + 四种经典错误 ——
def lb_forward(xy, lb):
    """原图坐标 -> letterbox 坐标。xy: (N,2) 或 (N,4) 的 xyxy"""
    xy = np.asarray(xy, float).copy()
    off = np.array([lb['left'], lb['top']] * (xy.shape[1]//2))
    return xy*lb['r'] + off

def lb_inverse(xy, lb, W0=W0, H0=H0):
    """letterbox 坐标 -> 原图坐标（**唯一正确的写法**）"""
    xy = np.asarray(xy, float).copy()
    off = np.array([lb['left'], lb['top']] * (xy.shape[1]//2))
    out = (xy - off) / lb['r']
    out[:, 0::2] = np.clip(out[:, 0::2], 0, W0)      # **clip 必须放在最后**
    out[:, 1::2] = np.clip(out[:, 1::2], 0, H0)
    return out

lb = LB['A']
r2 = np.random.default_rng(1)
pts = np.sort(r2.uniform(0, 1, (1000, 4)), axis=1) * np.array([W0, H0, W0, H0])
pts = pts[:, [0, 1, 2, 3]]
pts[:, 2] = np.maximum(pts[:, 2], pts[:, 0] + 1); pts[:, 3] = np.maximum(pts[:, 3], pts[:, 1] + 1)
assert np.abs(lb_inverse(lb_forward(pts, lb), lb) - pts).max() < 1e-6
print(f'round-trip 单测：1000 个随机框，最大误差 {np.abs(lb_inverse(lb_forward(pts,lb),lb)-pts).max():.2e}  ✅')

box_lb = lb_forward(np.array([[900., 400., 980., 480.]]), lb)     # 原图上一个 80x80 的牌子
right = lb_inverse(box_lb, lb)[0]
bad_nopad   = (box_lb / lb['r'])[0]                                # ❌ 忘了减 pad
bad_nonunif = (box_lb * np.array([W0/640, H0/640, W0/640, H0/640]))[0]   # ❌ 非等比还原
print(f"\n{'写法':<30s}{'还原出的框 (x1,y1,x2,y2)':>34s}{'y 方向偏移':>12s}")
print(f"{'✅ 正确 (x-pad)/r':<30s}{str(np.round(right,1)):>34s}{0.0:>12.1f}")
for nm, v in [('❌ x/r（忘了减 pad）', bad_nopad), ('❌ x·W0/640（非等比）', bad_nonunif)]:
    print(f'{nm:<30s}{str(np.round(v,1)):>34s}{v[1]-right[1]:>12.1f}')

assert np.allclose(right, [900, 400, 980, 480])
assert abs((bad_nopad[1] - right[1]) - lb['top']/lb['r']) < 1e-6
print(f"\n⚠️  忘了减 pad：y 整体偏 top/r = {lb['top']}/{lb['r']:.4f} = "
      f"{lb['top']/lb['r']:.0f} 像素 = 图高的 {lb['top']/lb['r']/H0:.0%}")
print('   识别法：**所有框偏移同一个常数 → 一定是坐标变换，不可能是模型**。')

def iou_shift(size, dx, dy):
    ix, iy = max(0, size-abs(dx)), max(0, size-abs(dy))
    inter = ix*iy
    return inter/(2*size*size - inter)
print(f'\n最难查的是 1 像素的系统性偏移（pad 取整不一致）。对一个 8×8 的标志：')
for dx, dy in [(1, 0), (2, 0), (2, 2)]:
    print(f'  平移 ({dx},{dy}) px -> IoU = {iou_shift(8, dx, dy):.4f}')
assert abs(iou_shift(8, 1, 0) - 56/72) < 1e-12
assert abs(iou_shift(8, 2, 0) - 48/80) < 1e-12
assert abs(iou_shift(8, 2, 2) - 36/92) < 1e-12
assert iou_shift(64, 2, 2) > 0.88, '同样 2px，64×64 的框几乎不受影响'
print(f'  对照：64×64 的框平移 (2,2) px，IoU 还有 {iou_shift(64,2,2):.4f}')
print('✅ 1px 的坐标 bug 在 COCO（大目标为主）上几乎不可见，在 TSR 上能吃掉 5–10 mAP。')

## 7 · BGR/RGB 与归一化：模型照跑，mAP 全掉

先看归一化的两个经典错误（都不会报错、都不会崩），再用一个合成检测任务量化代价。

In [ ]:
# —— ① 归一化顺序错：动态范围压缩正好 255 倍 ——
lo_ok,  hi_ok  = (0 - MEAN)/STD,        (255 - MEAN)/STD
lo_bad, hi_bad = (0/255 - MEAN)/STD,    (1.0 - MEAN)/STD
print('✅ 正确 (x-mean)/std      范围', np.round(lo_ok, 4), '~', np.round(hi_ok, 4),
      ' 跨度', np.round(hi_ok-lo_ok, 4))
print('❌ 先 /255 再用同一组参数 范围', np.round(lo_bad, 4), '~', np.round(hi_bad, 4),
      ' 跨度', np.round(hi_bad-lo_bad, 6))
ratio = (hi_ok-lo_ok)/(hi_bad-lo_bad)
print('\n压缩倍数 =', np.round(ratio, 6), ' → **与 mean/std 的具体取值无关，恒等于 255**')
assert np.allclose(ratio, 255.0)

# —— ② 在 uint8 上做减法：下溢回绕 ——
px  = np.array([10, 60, 124, 200], dtype=np.uint8)
sub = np.full(4, 124, dtype=np.uint8)
print('\nuint8 减法:', px, '-', sub, '=', px - sub)
assert (px - sub).tolist() == [142, 192, 0, 76]
print('⚠️  10-124 = -114 → mod 256 → **142**：图上最暗的像素变成了最亮的。')
print('   指纹：diff 只在暗区（像素 < mean）非零，且差值恰好是 256。')
print('   表现：白天正常、夜间全崩 —— 极易被误判成「夜间数据不够」（最贵的那条路）。')

# —— ③ 5 行 assert 挡住上面所有变体 ——
def sanity(t, name=''):
    t = np.asarray(t, float)
    ch = tuple(range(t.ndim-1)) if t.ndim == 3 else None
    ok = dict(std_ok=0.8 <= t.std() <= 1.2,
              mean_ok=abs(t.mean()) <= 0.5,
              range_ok=(t.min() > -3) and (t.max() < 3),
              ch_mean_ok=bool(np.all(np.abs(t.mean(axis=ch)) <= 0.5)))
    return all(ok.values()), ok

raw = np.stack([im, im*0.9, im*0.8], -1)
REF = (raw - MEAN)/STD                                   # 训练侧的参考张量
tests = {
    '✅ 正确':            REF,
    '❌ 先 /255':         (raw/255 - MEAN)/STD,
    '❌ 忘了归一化':       raw,
    '❌ 多除了一次 255':   (raw - MEAN)/STD/255,
    '❌ mean/std 反了序':  (raw - MEAN[::-1])/STD[::-1],
}
print(f"\n{'场景':<20s}{'std':>9s}{'mean':>9s}{'min':>9s}{'max':>9s}{'自检':>7s}{'对拍':>7s}")
for nm, t in tests.items():
    good, det = sanity(t)
    par = 'PASS' if np.abs(t - REF).max() <= 0.02 else 'FAIL'
    print(f'{nm:<20s}{t.std():>9.3f}{t.mean():>9.3f}{t.min():>9.2f}{t.max():>9.2f}'
          f'{str(good):>7s}{par:>7s}')
assert sanity(tests['✅ 正确'])[0]
for k in ['❌ 先 /255', '❌ 忘了归一化', '❌ 多除了一次 255']:
    assert not sanity(tests[k])[0], k
# **最隐蔽的一个：mean/std 顺序反了，四条自检全部通过**
assert sanity(tests['❌ mean/std 反了序'])[0], 'std/mean/range 都在正常区间里'
assert np.abs(tests['❌ mean/std 反了序'] - REF).max() > 0.02, '但对拍立刻发现'
print('\n✅ 三条自检（std / mean / range）5 行代码挡住：少除 255、多除 255、忘了归一化。')
print('⚠️  但最后一行 **四条自检全部通过**：mean/std 顺序反了时，std 正常、逐通道 mean')
print('   的偏移只有 0.3 左右，落在任何合理阈值内。**只有对拍能抓到它。**')
print('   这就是「自检 ≠ 对拍」：自检查的是「看起来正不正常」，对拍查的是「和训练侧一不一样」。')

### 7.1 四类合成 TSR 检测任务

- **限速60 / 限速80**：同为红环白面，只有内部细笔画不同 → **细粒度、依赖高频**
- **蓝色指示 / 黄色警告**：靠大色块区分 → **粗粒度、依赖颜色**

检测器刻意做成两段（模拟真实检测器）：
**定位靠灰度局部对比度（对通道置换完全不变）**，**分类靠归一化后的颜色特征（对通道置换极敏感）**。

In [ ]:
HI, LO, S = 288, 96, 3                # 高分辨率画布 -> 网络输入（3 倍下采样）
SGN_HI, SGN_LO = 36, 12               # 标志在两个尺度下的边长（12 px = TSR 的典型尺寸）
CLS = ['限速60(红)', '限速80(红)', '指示(蓝)', '警告(黄)']

def draw_sign(canvas, cx, cy, cls, r_):
    r = SGN_HI//2
    yy_, xx_ = np.mgrid[-r:r, -r:r]; d = np.hypot(xx_, yy_)
    p = np.zeros((SGN_HI, SGN_HI, 3)); m = d < r
    if cls in (0, 1):
        p[...] = [205., 35., 45.]; p[d < r*0.74] = [238., 238., 234.]
        for b in ([-9, 0, 9] if cls == 1 else [-6, 6]):        # 3px 宽笔画 = 低分辨率下 1px
            p[(np.abs(xx_-b) < 1.5) & (np.abs(yy_) < 10)] = [35., 35., 38.]
    elif cls == 2:
        p[...] = [25., 65., 175.]
        p[(np.abs(xx_) < 3) & (yy_ > -10)] = [240.]*3
        p[(np.abs(xx_) + np.abs(yy_+8) < 8) & (yy_ < -2)] = [240.]*3
    else:
        p[...] = [240., 200., 40.]
        p[(np.abs(xx_) < 2) & (yy_ > -6) & (yy_ < 8)] = [30.]*3
        m = (yy_ > -r*0.85) & (np.abs(xx_) < (yy_ + r*0.85)*0.62)
    y0, x0 = cy-r, cx-r
    reg = canvas[y0:y0+SGN_HI, x0:x0+SGN_HI]
    reg[m] = p[m] * r_.uniform(0.88, 1.08)
    return (x0, y0, x0+SGN_HI, y0+SGN_HI)

def make_scene(r_):
    y_, x_ = np.mgrid[0:HI, 0:HI]
    c = np.stack([110+40*np.sin(x_/47.), 118+35*np.cos(y_/39.), 125+30*np.sin((x_+y_)/61.)], -1)
    c = c + 18*np.sin(2*np.pi*x_/5.)[..., None] + 14*np.sin(2*np.pi*y_/4.)[..., None]
    for _ in range(4):                                          # 干扰块
        h, w = r_.integers(14, 40, 2); y, x = r_.integers(0, HI-40, 2)
        c[y:y+h, x:x+w] += r_.uniform(-55, 55, 3)
    cls = int(r_.integers(0, 4)); cx, cy = r_.integers(24, HI-24, 2)
    box = draw_sign(c, cx, cy, cls, r_)          # 必须先画再 clip（否则拿到的是空场景）
    return np.clip(c, 0, 255), box, cls

def prep(img, resize='area', swap=False, norm='ok'):
    """完整的预处理：resize -> (可选)通道置换 -> 归一化"""
    x = {'area': resize_area, 'antialias': resize_antialias}.get(
        resize, lambda a, b, c: resize_bilinear(a, b, c, 'half_pixel'))(img, LO, LO)
    if swap:
        x = x[..., ::-1]
    return (x - MEAN)/STD if norm == 'ok' else (x/255.0 - MEAN)/STD

srng = np.random.default_rng(60)
TRAIN = [make_scene(srng) for _ in range(60)]
TEST  = [make_scene(srng) for _ in range(100)]
print(f'训练 {len(TRAIN)} 张 / 测试 {len(TEST)} 张，每张 1 个标志，'
      f'高分辨率 {HI}×{HI} -> 网络输入 {LO}×{LO}（3 倍下采样，和 1920→640 同构）')
print('类别分布(测试):', np.bincount([c for _, _, c in TEST], minlength=4).tolist(), CLS)

In [ ]:
def windows(t, st=2, n=SGN_LO):
    ys = np.arange(0, LO-n+1, st); xs = np.arange(0, LO-n+1, st)
    Pw = t[ys[:, None]+np.arange(n)][:, :, xs[:, None]+np.arange(n)].transpose(0, 2, 1, 3, 4)
    B = np.stack(np.meshgrid(xs, ys), -1).reshape(-1, 2).astype(float)
    M = len(ys)*len(xs)
    return (Pw.reshape(M, -1),                                  # 分类特征
            Pw.mean(-1).reshape(M, n*n).std(1),                 # objectness：**灰度**局部对比度
            np.concatenate([B, B+n], 1))

def split_feat(F, w):
    """[ (1-w)·灰度 , w·色度 ]。灰度在通道置换下不变，色度会被置换 -> w 就是「颜色依赖度」。"""
    G = F.reshape(len(F), -1, 3); g = G.mean(-1)
    return np.concatenate([(1-w)*g, w*(G - g[..., None]).reshape(len(F), -1)], 1)

def protos_of(scenes, w, **kw):
    acc = {c: [] for c in range(4)}
    for img, box, cls in scenes:
        t = prep(img, **kw); x0, y0 = box[0]//S, box[1]//S
        acc[cls].append(t[y0:y0+SGN_LO, x0:x0+SGN_LO].ravel())
    return split_feat(np.array([np.mean(acc[c], 0) for c in range(4)]), w)

def detect(img, protos, w, tau=40.0, **kw):
    F, O, B = windows(prep(img, **kw)); F = split_feat(F, w)
    d2 = (F**2).sum(1)[:, None] - 2*F@protos.T + (protos**2).sum(1)[None]
    lg = -d2/(2*tau); lg -= lg.max(1, keepdims=True)
    p = np.exp(lg); p /= p.sum(1, keepdims=True)
    conf = (O/(O.max()+1e-9)) * p.max(1)                        # 定位×分类
    i = int(np.argmax(conf))
    return B[i], int(np.argmax(p[i])), float(conf[i])

def iou1(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    it = max(0, x2-x1)*max(0, y2-y1)
    return it/((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - it + 1e-9)

def voc_ap(rec, pre):
    mr = np.concatenate([[0], rec, [1]]); mp = np.concatenate([[0], pre, [0]])
    for i in range(len(mp)-2, -1, -1):
        mp[i] = max(mp[i], mp[i+1])
    k = np.where(mr[1:] != mr[:-1])[0]
    return float(((mr[k+1]-mr[k])*mp[k+1]).sum())

def evaluate(scenes, protos, w, **kw):
    dets = {c: [] for c in range(4)}; npos = {c: 0 for c in range(4)}
    pred, loc_ok = [], []
    for img, box, cls in scenes:
        npos[cls] += 1
        gtb = (box[0]/S, box[1]/S, box[2]/S, box[3]/S)
        pb, pc, sc = detect(img, protos, w, **kw)
        pred.append(pc); loc_ok.append(iou1(pb, gtb) >= 0.5)     # **类别无关**的定位召回
        dets[pc].append((sc, (pc == cls) and iou1(pb, gtb) >= 0.5))
    aps = []
    for c in range(4):
        d = sorted(dets[c], key=lambda z: -z[0])
        if not d:
            aps.append(0.0); continue
        tp = np.cumsum([1 if o else 0 for _, o in d]); fp = np.cumsum([0 if o else 1 for _, o in d])
        aps.append(voc_ap(tp/max(npos[c], 1), tp/np.maximum(tp+fp, 1e-9)))
    return dict(mAP=float(np.mean(aps)), AP=aps,
                loc=float(np.mean(loc_ok)), pred=np.bincount(pred, minlength=4))

W_COLOR = 0.5
PROTOS = protos_of(TRAIN, W_COLOR, resize='area')     # 「训练侧」用 AREA + RGB + 正确归一化
print('原型就位（4 类 × %d 维），训练侧预处理 = AREA + RGB + (x-mean)/std' % PROTOS.shape[1])

In [ ]:
t0 = time.time()
CONDS = [('① 完全一致（基线）',      dict(resize='area')),
         ('② AREA→INTER_LINEAR',   dict(resize='linear')),
         ('③ AREA→抗锯齿 bilinear', dict(resize='antialias')),
         ('④ BGR/RGB 弄反',        dict(resize='area', swap=True)),
         ('⑤ 归一化顺序错',         dict(resize='area', norm='bad')),
         ('⑥ ②+④ 叠加',            dict(resize='linear', swap=True))]
R = {}
print(f"{'条件':<24s}{'mAP':>7s}{'定位召回':>9s}   逐类 AP  " + ' / '.join(CLS))
for nm, kw in CONDS:
    R[nm[0]] = evaluate(TEST, PROTOS, W_COLOR, **kw)
    r_ = R[nm[0]]
    print(f"{nm:<24s}{r_['mAP']:>7.3f}{r_['loc']:>9.2f}   " +
          '  '.join(f'{a:.2f}' for a in r_['AP']))
print(f'（{time.time()-t0:.1f}s）')

b, l, a2, sw, nb = R['①'], R['②'], R['③'], R['④'], R['⑤']
assert b['mAP'] > 0.70, b['mAP']
# ② resize 不一致：整体掉幅温和，但**全部集中在两个细粒度红牌类**
d_fine   = (b['AP'][0]+b['AP'][1]) - (l['AP'][0]+l['AP'][1])
d_coarse = (b['AP'][2]+b['AP'][3]) - (l['AP'][2]+l['AP'][3])
print(f"\n② resize 不一致: 整体 mAP {b['mAP']:.3f} → {l['mAP']:.3f}（−{b['mAP']-l['mAP']:.3f}，"
      f'很容易被当成种子噪声放过）')
print(f'   但两个**细粒度红牌类**的 AP 合计掉了 {d_fine:.2f}，'
      f'两个靠颜色区分的粗类合计变化 {-d_coarse:+.2f}')
assert d_fine > 0.20 and abs(d_coarse) < 0.12, (d_fine, d_coarse)
assert abs(a2['mAP'] - b['mAP']) < 0.03, '两种都抗锯齿的实现之间几乎无差'
# ④ BGR：定位完好、类别全错
print(f"\n④ BGR 弄反: mAP {sw['mAP']:.3f}，但**类别无关定位召回仍有 {sw['loc']:.2f}** —— "
      f'框全在，只是类别全错')
print(f"   预测类别分布 {sw['pred'].tolist()} vs 真值 "
      f"{np.bincount([c for _,_,c in TEST], minlength=4).tolist()}"
      f" → {sw['pred'].max()}/100 全被判成「{CLS[int(sw['pred'].argmax())]}」")
assert sw['mAP'] < 0.05 and sw['loc'] > 0.8, (sw['mAP'], sw['loc'])
# ⑤ 归一化顺序错：输出塌缩到单一类别
print(f"\n⑤ 归一化顺序错: 预测类别分布 {nb['pred'].tolist()} —— 全部塌缩到一个类")
assert nb['pred'].max() >= 90
print('\n✅ 「整体指标掩盖关键类崩塌」+「框全在但类别全错」= 预处理 bug「最隐蔽」的两种形态。')

In [ ]:
# —— 颜色依赖度 w 扫描：BGR 弄反的代价取决于模型多依赖颜色 ——
t0 = time.time()
print(f"{'w (颜色依赖度)':>14s}{'正确 RGB mAP':>14s}{'BGR 弄反 mAP':>14s}{'Δ':>9s}")
deltas = []
for w in [0.0, 0.2, 0.5, 0.8]:
    Pw = protos_of(TRAIN, w, resize='area')
    m0 = evaluate(TEST, Pw, w, resize='area')['mAP']
    m1 = evaluate(TEST, Pw, w, resize='area', swap=True)['mAP']
    deltas.append(m0-m1)
    print(f'{w:>14.1f}{m0:>14.3f}{m1:>14.3f}{m0-m1:>+9.3f}')
print(f'（{time.time()-t0:.1f}s）')
assert abs(deltas[0]) < 0.02, 'w=0 时特征完全是灰度的，通道置换必须毫无影响'
assert deltas[-1] > 0.5 and deltas[1] > 0.3
assert deltas[1] < deltas[-1], '颜色依赖度越高，代价越大'
print('\n✅ 结论：**模型对颜色的依赖度越高，通道顺序错误的代价越大。**')
print('   而 TSR 是所有视觉任务里颜色依赖度最高的之一 —— 红=禁令、蓝=指示、黄=警告，')
print('   **颜色就是语义**（C56 模块 02）。所以 BGR/RGB 弄反在通用检测里是中等 bug，')
print('   在 TSR 里是致命 bug。合成实验掉到 0 是因为分类器纯靠颜色；')
print('   真实 CNN 还会用形状与纹理，典型实际掉幅是 **10–30 mAP**，不会归零。')

## 8 · 预处理对拍工具

把预处理写成**命名阶段的序列**（而不是一个 80 行的大函数），
然后：逐阶段 diff → 二分定位首个分歧 → diff 指纹分类 → 输出定位报告。

In [ ]:
STAGES = ['to_rgb', 'resize', 'letterbox_pad', 'normalize', 'hwc2chw']

def OPS(name, t, cfg):
    if name == 'to_rgb':
        return t[..., ::-1] if cfg.get('swap_channels') else t
    if name == 'resize':
        k = cfg.get('resize', 'area')
        return {'area': resize_area, 'antialias': resize_antialias}.get(
            k, lambda a, b, c: resize_bilinear(a, b, c, 'half_pixel'))(t, LO, LO)
    if name == 'letterbox_pad':
        p = cfg.get('pad', 0)
        return np.pad(t, ((p, p), (p, p), (0, 0)), constant_values=cfg.get('pad_value', 114))
    if name == 'normalize':
        return (t/255.0 - MEAN)/STD if cfg.get('norm') == 'bad' else (t - MEAN)/STD
    if name == 'hwc2chw':
        return np.ascontiguousarray(np.moveaxis(t, -1, 0))
    raise ValueError(name)

def run_pipeline(img, cfg):
    t, out = np.asarray(img, float), {}
    for name in STAGES:
        t = OPS(name, t, cfg); out[name] = t
    return out

def fingerprint(a, b, tol=2e-3):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if a.shape != b.shape:
        return 'shape_mismatch'
    d = b - a
    if np.abs(d).max() <= tol:
        return 'noise'
    ch = lambda z: float(np.mean(np.std(z, axis=tuple(range(z.ndim-1)) if z.ndim == 3 else None)))
    if ch(b) < 0.1*ch(a):                                        # **逐通道**看，不能看整体
        return 'range_collapse'                                  # 归一化顺序错的专属指纹
    if a.ndim == 3 and a.shape[-1] == 3 and np.abs(b[..., ::-1] - a).max() <= tol:
        return 'channel_swap'
    e = np.zeros(a.shape[:2], bool)
    if min(a.shape[:2]) > 6:                                     # 太小的图不做边缘带判定
        e[:2] = e[-2:] = True; e[:, :2] = e[:, -2:] = True
        de, di = np.abs(d[e]).mean(), np.abs(d[~e]).mean()
        if di < 1e-9 or de/max(di, 1e-12) > 20:
            return 'border'
    ax = tuple(range(d.ndim-1)) if a.ndim == 3 else None
    if np.abs(d - d.mean(axis=ax, keepdims=True)).max() < 0.05*np.abs(d).max() + 1e-9:
        return 'const_shift'
    hf = np.abs(np.diff(d, axis=1)).mean()/(np.abs(d).mean() + 1e-12)
    return 'high_freq' if hf > 0.8 else 'unknown'

CAUSE = {'high_freq': '插值核不同（缺少抗锯齿）→ 查 resize 的 interpolation',
         'channel_swap': 'BGR/RGB 弄反 → 查 cvtColor / ISP 输出格式',
         'range_collapse': '归一化顺序错（先 /255 又减了 0-255 量纲的 mean）',
         'border': 'align_corners / padding 值或位置不同',
         'const_shift': 'mean 不同 / uint8→float 时机不同',
         'shape_mismatch': '形状就不同 —— 查 letterbox 的 auto / 目标尺寸',
         'noise': '数值噪声，不是 bug', 'unknown': '未知，需人工看 diff'}

def bisect_first(probe, n):
    lo, hi, calls = 0, n+1, 0
    while hi - lo > 1:
        mid = (lo+hi)//2; calls += 1
        hi, lo = (mid, lo) if probe(mid) else (hi, mid)
    return (None if hi == n+1 else hi), calls

def parity(img, cfg_l, cfg_r, tol_max=0.02, tol_mean=0.005, verbose=True):
    L, Rr = run_pipeline(img, cfg_l), run_pipeline(img, cfg_r)
    rows = []
    for name in STAGES:
        a, b = np.asarray(L[name], float), np.asarray(Rr[name], float)
        if a.shape != b.shape:
            rows.append((name, str(a.shape), np.inf, np.inf, 'FAIL', 'shape_mismatch')); continue
        d = np.abs(a-b); sc = STD.mean() if name in ('to_rgb', 'resize', 'letterbox_pad') else 1.0
        ok = (d.max()/sc <= tol_max) and (d.mean()/sc <= tol_mean)
        rows.append((name, str(a.shape), d.max(), d.mean(),
                     'PASS' if ok else 'FAIL', fingerprint(a, b) if not ok else '-'))
    k, calls = bisect_first(lambda i: rows[i-1][4] == 'FAIL', len(STAGES))
    if verbose:
        print(f"  {'阶段':<14s}{'形状':>18s}{'max|Δ|':>11s}{'mean|Δ|':>11s}{'判定':>7s}  指纹")
        for nm, sh, mx, mn, v, fp in rows:
            print(f'  {nm:<14s}{sh:>18s}{mx:>11.4f}{mn:>11.4f}{v:>7s}  {fp}')
        if k is None:
            print('  ══ 全部阶段一致 ✅')
        else:
            fp = rows[k-1][5]
            print(f'  ══ 首个分歧阶段: **{STAGES[k-1]}**（二分探针 {calls} 次，'
                  f'上界 ⌈log2 {len(STAGES)+1}⌉={math.ceil(math.log2(len(STAGES)+1))}）')
            print(f'  ══ 指纹: {fp} → {CAUSE[fp]}')
    return (STAGES[k-1] if k else None), (rows[k-1][5] if k else None), rows

BASE = dict(resize='area', pad=0, norm='ok')
img0 = TEST[0][0]
for title, cfg in [('注入 bug ①：部署侧用了 INTER_LINEAR', dict(BASE, resize='linear')),
                   ('注入 bug ②：部署侧漏了 BGR→RGB',      dict(BASE, swap_channels=True)),
                   ('注入 bug ③：部署侧先除了 255',         dict(BASE, norm='bad')),
                   ('对照：两侧完全一致',                    dict(BASE))]:
    print(f'\n【{title}】')
    st, fp, _ = parity(img0, BASE, cfg)

st1, fp1, _ = parity(img0, BASE, dict(BASE, resize='linear'), verbose=False)
st2, fp2, _ = parity(img0, BASE, dict(BASE, swap_channels=True), verbose=False)
st3, fp3, _ = parity(img0, BASE, dict(BASE, norm='bad'), verbose=False)
st4, fp4, _ = parity(img0, BASE, dict(BASE, pad=8), verbose=False)
st0, fp0, _ = parity(img0, BASE, dict(BASE), verbose=False)
assert (st1, fp1) == ('resize', 'high_freq'), (st1, fp1)
assert (st2, fp2) == ('to_rgb', 'channel_swap'), (st2, fp2)
assert (st3, fp3) == ('normalize', 'range_collapse'), (st3, fp3)
assert (st4, fp4) == ('letterbox_pad', 'shape_mismatch'), (st4, fp4)
assert (st0, fp0) == (None, None)
print('\n✅ 四种注入的 bug 全部被定位到正确的阶段并给出正确的指纹；对照组全绿。')
print('   注意 bug ① 的 diff 在 normalize 之后从 21 灰阶变成 0.37 —— ')
print('   **归一化会把差异同时除以 σ≈58，让巨大的差异「看起来不大」。要在灰阶量纲上看 diff。**')

## ✏️ 练习 1：INTER_AREA 的权重矩阵

实现 `area_weights(out_size, in_size)`，返回形状 `(out_size, in_size)` 的矩阵：
输出像素 `j` 覆盖源区间 `[j·s, (j+1)·s)`（`s = in_size/out_size`），
权重 = **重叠长度**，每行归一化到和为 1。

In [ ]:
def area_weights(out_size, in_size):
    # TODO: 对每个 j 算区间 [j*s, (j+1)*s) 与每个源像素 [i, i+1) 的重叠长度
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测（可手算）——
# 6 -> 2：s=3，两个输出像素各自平均 3 个源像素
assert np.allclose(area_weights(2, 6), [[1/3, 1/3, 1/3, 0, 0, 0], [0, 0, 0, 1/3, 1/3, 1/3]])
# 3 -> 2：s=1.5，j=0 覆盖 [0,1.5) -> 权重 [1, 0.5, 0]/1.5 = [2/3, 1/3, 0]
assert np.allclose(area_weights(2, 3), [[2/3, 1/3, 0], [0, 1/3, 2/3]])
# 恒等：out == in 时必须是单位阵
assert np.allclose(area_weights(5, 5), np.eye(5))
for o, i in [(3, 7), (16, 40), (7, 96), (2, 11)]:
    W = area_weights(o, i)
    assert W.shape == (o, i)
    assert np.allclose(W.sum(1), 1.0), (o, i)
    assert (W >= 0).all()
    assert np.allclose(W, _area_w(o, i)), (o, i)                 # 与正文实现一致
# 上采样时 AREA 退化成最近邻（每行只有一个非零）
assert (np.count_nonzero(area_weights(6, 3), axis=1) == 1).all()
print('6→2 的权重:\n', area_weights(2, 6))
print('3→2 的权重:\n', np.round(area_weights(2, 3), 4))
print('✅ 练习 1 通过：AREA 是唯一「支撑集随缩放比拉伸」的核，也是缩小时唯一正确的选择。')

## ✏️ 练习 2：letterbox 的逆变换

实现 `unletterbox(boxes_lb, r, left, top, W0, H0)`：
把 letterbox 坐标系下的 `(N,4)` xyxy 框还原回原图坐标，并 clip 到原图范围。
**三个考点**：① 先减 pad 再除 r（不是先除）② 用同一个 r（不是 W0/target 和 H0/target）
③ **clip 放在最后**（clip 的边界是原图的，不是 letterbox 的）。

In [ ]:
def unletterbox(boxes_lb, r, left, top, W0, H0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
LBP = letterbox(1080, 1920, 640, center=True)      # r=1/3, left=0, top=140
r_, l_, t_ = LBP['r'], LBP['left'], LBP['top']
# ① round-trip：1000 个随机框
rr = np.random.default_rng(2)
gt = np.sort(rr.uniform(0, 1, (1000, 4)), 1) * np.array([1920, 1080, 1920, 1080])
fwd = gt*r_ + np.array([l_, t_, l_, t_])
assert np.abs(unletterbox(fwd, r_, l_, t_, 1920, 1080) - gt).max() < 1e-6
# ② 一个手算的例子：原图 (900,400,980,480)
one = np.array([[900., 400., 980., 480.]])*r_ + np.array([l_, t_, l_, t_])
assert np.allclose(one, [[300., 273.3333, 326.6667, 300.]], atol=1e-3), one
assert np.allclose(unletterbox(one, r_, l_, t_, 1920, 1080), [[900, 400, 980, 480]])
# ③ clip 必须在最后：一个越出上边界的框，正确结果是 y1 被 clip 到 0（而不是负值或被截错）
edge = np.array([[10., 100., 60., 200.]])          # y=100 < top=140 -> 原图上是负的
out = unletterbox(edge, r_, l_, t_, 1920, 1080)
assert out[0, 1] == 0.0 and abs(out[0, 3] - (200-140)/r_) < 1e-6, out
# ④ 常见错误必须被区分开
wrong_nopad = edge/r_
assert abs(wrong_nopad[0, 1] - out[0, 1]) > 100, '忘了减 pad 会差几百像素'
print('还原结果:', np.round(unletterbox(one, r_, l_, t_, 1920, 1080), 2))
print('越界框 clip 后:', np.round(out, 2))
print('✅ 练习 2 通过：把正/逆变换写成一对函数 + round-trip 单测，10 行代码挡住四种经典错误。')

## ✏️ 练习 3：混叠判据——多细的笔画会被毁掉

实现 `aliasing_report(stroke_px, scale)`，`stroke_px` 是**原图上**的笔画宽度：

- `period_src = 2 * stroke_px`（一条亮线 + 一条暗线 = 一个周期）
- `nyquist_period_src = 2 * scale`（输出端能表示的最细周期，换算到原图坐标）
- `aliased = period_src < nyquist_period_src`
- 折叠后的输出周期：`f = 1/period_src`（cyc/源像素）→ `fs = f*scale`（cyc/输出像素）
  → `folded = min(fs % 1, 1 - fs % 1)` → `folded_period_out = 1/folded`（`folded==0` 时记为 `inf`）

返回 `dict(period_src, nyquist_period_src, aliased, folded_period_out)`。

In [ ]:
def aliasing_report(stroke_px, scale):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r1 = aliasing_report(2, 3)          # 2px 笔画，3 倍下采样
assert r1['period_src'] == 4 and r1['nyquist_period_src'] == 6
assert r1['aliased'] is True
assert abs(r1['folded_period_out'] - 4.0) < 1e-9, r1
r2_ = aliasing_report(3, 3)         # 3px 笔画：恰好在临界
assert r2_['aliased'] is False and abs(r2_['folded_period_out'] - 2.0) < 1e-9
assert aliasing_report(6, 3)['aliased'] is False
assert aliasing_report(1, 3)['aliased'] is True
assert aliasing_report(4, 2)['aliased'] is False and aliasing_report(1, 2)['aliased'] is True
# 安全下界：笔画宽度必须 >= scale
for sc in [2, 3, 4]:
    assert aliasing_report(sc, sc)['aliased'] is False
    assert aliasing_report(sc-0.5, sc)['aliased'] is True
print(f"{'原图笔画宽':>11s}{'周期':>7s}{'可表示?':>9s}{'折叠后的输出周期':>18s}")
for st in [1, 2, 3, 4, 6, 10]:
    r_rep = aliasing_report(st, 3)
    print(f"{st:>11.1f}{r_rep['period_src']:>7.1f}"
          f"{('❌ 混叠' if r_rep['aliased'] else '✅ 安全'):>10s}"
          f"{r_rep['folded_period_out']:>18.2f}")
print('\n✅ 练习 3 通过：**1920→640（s=3）时，原图上笔画宽度 < 3px 的一切都是假的。**')
print('   60 米外的限速牌约 20–30 px，「60」的笔画宽度正好 2–3 px —— 正踩在临界上。')
print('   这就是「为什么 TSR 对 resize 尤其敏感」的定量回答（面试可直接用）。')

## ✏️ 练习 4：对拍定位器

实现 `locate_bug(img, cfg_l, cfg_r, tol_max=0.02, tol_mean=0.005)`，
返回 `(首个分歧阶段名 或 None, 指纹 或 None)`：

1. 两侧各跑一遍 `run_pipeline`
2. 逐阶段比对（形状不同 → 立即 `'shape_mismatch'`）
3. 容差：`to_rgb/resize/letterbox_pad` 三个阶段的 diff 要**除以 σ≈58** 换算到 tensor 单位再比；
   `normalize/hwc2chw` 已经是 tensor 单位，直接比
4. 首个 FAIL 的阶段用 `fingerprint` 给出指纹

In [ ]:
def locate_bug(img, cfg_l, cfg_r, tol_max=0.02, tol_mean=0.005):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
CASES = [(dict(BASE, resize='linear'),      ('resize', 'high_freq')),
         (dict(BASE, swap_channels=True),   ('to_rgb', 'channel_swap')),
         (dict(BASE, norm='bad'),           ('normalize', 'range_collapse')),
         (dict(BASE, pad=8),                ('letterbox_pad', 'shape_mismatch')),
         (dict(BASE),                       (None, None))]
for cfg, want in CASES:
    got = locate_bug(img0, BASE, cfg)
    assert got == want, (cfg, got, want)
    print(f'{str({k: v for k, v in cfg.items() if BASE.get(k) != v}):<32s} -> {got}')
# 换一张图、换一个 bug 组合也必须稳定
assert locate_bug(TEST[7][0], BASE, dict(BASE, resize='linear', swap_channels=True)) == \
       ('to_rgb', 'channel_swap'), '两个 bug 叠加时，报**最早**的那个'
# 定点舍入这一档不能报警（实现噪声 ≠ 语义错误）
noisy = dict(BASE)
img_noisy = np.clip(img0 + rng.uniform(-0.5, 0.5, img0.shape), 0, 255)
st, fp = locate_bug(img0, BASE, noisy)
assert (st, fp) == (None, None)
print('\n✅ 练习 4 通过：一个 20 行的函数，把「猜两周」变成「30 秒给出根因」。')
print('   把它接到 CI 上（pytest），预处理不一致这个占 40% 的根因就会绝迹。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def area_weights(out_size, in_size):
    s = in_size / out_size
    W = np.zeros((out_size, in_size))
    for j in range(out_size):
        lo, hi = j*s, (j+1)*s
        for i in range(int(np.floor(lo)), min(int(np.ceil(hi)), in_size)):
            W[j, i] = max(0.0, min(hi, i+1) - max(lo, i))
    return W / W.sum(1, keepdims=True)

In [ ]:
# 练习 2 参考答案
def unletterbox(boxes_lb, r, left, top, W0, H0):
    b = np.asarray(boxes_lb, float).copy()
    b[:, 0::2] = (b[:, 0::2] - left) / r          # ① 先减 pad 再除 r
    b[:, 1::2] = (b[:, 1::2] - top) / r           # ② 两个方向用**同一个** r
    b[:, 0::2] = np.clip(b[:, 0::2], 0, W0)       # ③ clip 放在最后
    b[:, 1::2] = np.clip(b[:, 1::2], 0, H0)
    return b

In [ ]:
# 练习 3 参考答案
def aliasing_report(stroke_px, scale):
    period_src = 2.0 * stroke_px
    nyq = 2.0 * scale
    f = 1.0 / period_src                          # cyc / 源像素
    fs = f * scale                                # cyc / 输出像素
    frac = fs % 1.0
    folded = min(frac, 1.0 - frac)
    return dict(period_src=period_src, nyquist_period_src=nyq,
                aliased=bool(period_src < nyq),
                folded_period_out=(float('inf') if folded == 0 else 1.0/folded))

In [ ]:
# 练习 4 参考答案
def locate_bug(img, cfg_l, cfg_r, tol_max=0.02, tol_mean=0.005):
    L, Rr = run_pipeline(img, cfg_l), run_pipeline(img, cfg_r)
    for name in STAGES:
        a, b = np.asarray(L[name], float), np.asarray(Rr[name], float)
        if a.shape != b.shape:
            return name, 'shape_mismatch'
        sc = STD.mean() if name in ('to_rgb', 'resize', 'letterbox_pad') else 1.0
        d = np.abs(a - b)
        if d.max()/sc > tol_max or d.mean()/sc > tol_mean:
            return name, fingerprint(a, b)
    return None, None

---
## 🧪 真实工程胶囊：预处理规格 + 对拍脚本 + CI 门禁

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 一、预处理规格（两侧都从它生成，不要各写各的）—— preprocess.yaml
# ══════════════════════════════════════════════════════════════════════
preprocess:
  version: 3                       # **改任何一项都要 +1，并记进模型卡**
  channel_order: RGB               # 输入张量的通道顺序（不是解码器的顺序！）
  source_order: BGR                # cv2.imread 的输出；ISP 直出时改成 RGB 并**删掉 cvtColor**
  resize:
    target: [640, 640]
    interpolation: INTER_AREA      # ← **缩小必须用 AREA**（或 antialias=True）
    coordinate_transformation: half_pixel
  letterbox:
    pad_value: 114
    center: true
    pad_to: stride                 # stride | square
    stride: 32
    scaleup: false                 # 推理端通常不放大小图
  dtype: {cast_before_resize: true, cast_to: float32}
  normalize:
    mean: [123.675, 116.28, 103.53]   # **0-255 量纲**
    std:  [58.395, 57.12, 57.375]
    divide_255_first: false           # ← 与上面两行必须配套，改一个就要改另一个
  layout: NCHW

# ══════════════════════════════════════════════════════════════════════
# 二、各家库的正确写法（照抄）
# ══════════════════════════════════════════════════════════════════════
# OpenCV（缩小）：**必须显式写 INTER_AREA**，默认的 INTER_LINEAR 不抗锯齿
#   img = cv2.imread(p)                            # BGR, uint8
#   img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)     # ← 漏了它 = mAP 掉 10~30
#   img = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
#
# PIL：默认就抗锯齿（缩小时），坐标语义 = half_pixel
#   im = Image.open(p).convert('RGB').resize((w, h), Image.BILINEAR)
#
# torch：**默认不抗锯齿**，要显式打开
#   x = F.interpolate(x, size=(h, w), mode='bilinear',
#                     align_corners=False, antialias=True)
#   # torchvision.transforms.v2.Resize(..., antialias=True)  # 新版默认 True
#
# ONNX 导出后必查（TensorRT 的 IResizeLayer 默认是 ASYMMETRIC，和 PyTorch 不同！）
#   $ polygraphy inspect model model.onnx --show layers \
#       | grep -A3 -i resize | grep -i "coordinate_transformation_mode\|mode\|antialias"

# ══════════════════════════════════════════════════════════════════════
# 三、黄金样本集（必须刻意设计覆盖度，一张「正常图」什么都测不出来）
# ══════════════════════════════════════════════════════════════════════
#   golden/01_small_signs.png      含 <16px 的标志      -> 暴露 resize / 混叠
#   golden/02_overexposed.png      大面积过曝           -> 暴露 clip / 溢出
#   golden/03_odd_size_1919x1079   奇数尺寸             -> 暴露取整 / pad 边界
#   golden/04_night_dark.png       极暗帧               -> 暴露 uint8 下溢
#   golden/05_solid_color.png      纯色                 -> 反向验证（此时很多 bug 会互相抵消）
#   golden/06_16x9_edge_boxes.png  贴边的框             -> 暴露逆变换 / clip 顺序
#   golden/07_red_blue_signs.png   红蓝标志各半         -> 暴露 BGR/RGB
#   连同两侧的期望中间张量一起入库（几 MB），随模型一起发布给下游集成方。

# ══════════════════════════════════════════════════════════════════════
# 四、CI 门禁 —— tests/test_preproc_parity.py
# ══════════════════════════════════════════════════════════════════════
#   import numpy as np, pytest, glob
#   TOL = dict(max=0.02, mean=0.005)          # tensor 单位；来源见模块 00 容差表
#   STAGES = ['to_rgb','resize','letterbox','cast_f32','normalize','hwc2chw']
#
#   @pytest.mark.parametrize('img', sorted(glob.glob('golden/*.png')))
#   @pytest.mark.parametrize('stage', STAGES)
#   def test_stage_parity(img, stage):
#       a = np.load(f'dump/train/{stem(img)}_{stage}.npy')
#       b = np.load(f'dump/deploy/{stem(img)}_{stage}.npy')
#       assert a.shape == b.shape, f'SHAPE {a.shape} vs {b.shape}'
#       d = np.abs(a.astype(np.float64) - b.astype(np.float64))
#       assert d.max() <= TOL['max'] and d.mean() <= TOL['mean'], \
#              f'{stage}: max={d.max():.4f} mean={d.mean():.4f}'
#
#   def test_letterbox_roundtrip():            # 10 行，挡住四种经典逆变换错误
#       boxes = rng.uniform(0,1,(1000,4)) * [W0,H0,W0,H0]
#       assert np.abs(unletterbox(letterbox_fwd(boxes)) - boxes).max() < 1e-6
#
#   def test_normalize_sanity():               # 5 行，挡住 3 种归一化错误
#       t = preprocess(golden[0])
#       assert 0.8 <= t.std() <= 1.2 and abs(t.mean()) <= 0.5
#       assert t.min() > -3 and t.max() < 3
#
#   触发时机：改预处理代码 / 换 OpenCV 或 Pillow 版本 / 改 C++ 管线 / 换硬件平台。
#   **一致性不是靠小心，是靠门禁。**

# ══════════════════════════════════════════════════════════════════════
# 五、TSR 专项检查
# ══════════════════════════════════════════════════════════════════════
#  · 1920→640 是 s=3：INTER_LINEAR 在这里**退化成最近邻**，8/9 的像素被丢弃。必须用 AREA。
#  · 评测必须**按像素尺寸分桶**（<16 / 16-32 / 32-64 / >64）。整体 mAP 会掩盖细粒度类崩塌：
#    本 notebook 实测「整体只掉 0.07，两个限速类合计掉 0.32」。
#  · 逐类 AP 必看：BGR 弄反时定位召回几乎不变（0.90），只有类别全错 —— 可视化根本看不出来。
#  · 训练侧如果也用了无抗锯齿 resize：那是**正确性**问题不是一致性问题，
#    要改训练配方并重训，不能只改部署侧（否则反而更不一致）。
'''
print(RECIPE)
for tok in ['INTER_AREA', 'antialias=True', 'ASYMMETRIC', 'divide_255_first',
            'golden/03_odd_size', 'test_letterbox_roundtrip', '分桶', 's=3']:
    assert tok in RECIPE, tok
print('✅ 胶囊覆盖：规格文件 / 三家库正确写法 / 黄金样本覆盖度 / CI 门禁 / TSR 专项')

### 小结

- **预处理是整条管线里唯一被写了两遍、且没人看它输出的部分**——这就是它占 40% 根因的原因。
  七个岔路口的组合数约 1.2×10⁵，其中与训练侧一致的只有 1 种。
- **resize 有两个独立自由度**：坐标映射（half_pixel / align_corners / **asymmetric ← TensorRT 默认**）
  与插值核（nearest / bilinear / area / 抗锯齿）。前者的 diff 集中在边缘带，后者是高频条纹。
- **整数奇数倍缩放时，half_pixel 下的 bilinear 逐位等于最近邻**：`u(j)=s·j+(s−1)/2`。
  而 **1920→640 与 1080→360 都是 s=3**——车端最常见的那条预处理，
  丢掉了 **88.9%** 的像素，且这是可以逐位验证的事实，不是近似说法。
- **混叠对小目标是灾难**：3 倍下采样后原图上周期 <6px（笔画 <3px）的结构全是假的。
  实测周期 4px 的条纹以 **100% 幅度**穿过无抗锯齿 bilinear（AREA 压到 33，抗锯齿核压到 18）。
  自然图上 **无抗锯齿 vs AREA 平均差 21.4 灰阶 = 0.37 tensor 单位**；
  而两种抗锯齿实现之间只差 8.5 灰阶——**分水岭是「有没有抗锯齿」，不是「哪家库」**。
- **letterbox 有六个自由度**。方形 pad 时 **43.75% 的算力在算灰边**（16:9 输入），
  换成 stride 倍数 pad 省 37.5%。逆变换忘了减 pad → 框整体偏 **420 像素**；
  pad 取整差 1 像素 → 8×8 的框 IoU 从 1.0 掉到 **0.778**（COCO 上看不见，TSR 上吃掉 5–10 mAP）。
- **BGR/RGB 弄反：框全在，类别全错。** 实测定位召回仍有 0.90，而 mAP 归零；
  颜色依赖度越高代价越大——**TSR 里颜色就是语义，所以它在这里是致命 bug 而不是中等 bug。**
- **归一化顺序错让动态范围压缩恰好 255 倍**，输出塌缩到单一类别。
  三条自检（std∈[0.8,1.2]、|mean|≤0.5、range⊂(−3,3)）能挡住多数变体，
  但 **mean/std 顺序反了时四条自检全部通过——只有对拍能抓到**。
- **定位方法**：命名阶段 → 逐阶段 dump → 二分（⌈log₂(n+1)⌉ 次）→ diff 指纹 → 根因。
  **在灰阶量纲上看 diff**，因为归一化会把 21 灰阶变成 0.37，让巨大的差异「看起来不大」。
- **一张图对拍通过 ≠ 一致**：取整分歧只在奇数尺寸暴露、uint8 下溢只在暗像素暴露、
  非等比逆变换只在非正方形输入暴露。**黄金样本的覆盖度必须是刻意设计的。**

下一站：**模块 02 · TensorRT 构建与优化** —— 预处理对齐之后，才轮到讨论 engine。